[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/crewai/crewai_prompt_chaining_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 CrewAI - 提示词链工作流</h1>

In [ ]:
%pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp crewai crewai_tools openinference-instrumentation-crewai

# 设置密钥与依赖

注意：运行此 Colab 需要：

*   OpenAI API 密钥 (https://openai.com/)
*   Serper API 密钥 (https://serper.dev/)
*   Phoenix API 密钥 (https://app.phoenix.arize.com/)

In [ ]:
import os
from getpass import getpass

# 尚未设置 API 密钥时提示用户输入
openai_key = os.getenv("OPENAI_API_KEY", "OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY", "SERPER_API_KEY")

if openai_key == "OPENAI_API_KEY":
    openai_key = getpass("请输入 OPENAI_API_KEY： ")

if serper_key == "SERPER_API_KEY":
    serper_key = getpass("请输入 SERPER_API_KEY： ")

# 使用提供的密钥设置环境变量
os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="crewai-agents")

# 为 CrewAI 接入插桩

In [ ]:
from openinference.instrumentation.crewai import CrewAIInstrumentor

CrewAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 定义智能体

In [ ]:
from crewai import Agent, Crew, Task
from crewai.process import Process

research_analyst = Agent(
    role="资深研究分析师",
    goal="研究前沿 AI 主题并总结三大趋势。",
    backstory="AI 研究与趋势分析专家。",
    verbose=True,
)

content_strategist = Agent(
    role="技术内容策略师",
    goal="根据研究创建结构化文章提纲。",
    backstory="擅长设计引人入胜提纲的技术叙事者。",
    verbose=True,
)

content_reviewer = Agent(
    role="内容评审者",
    goal="验证提纲的清晰度、语气和完整性。",
    backstory="注重技术准确性的编辑专家。",
    verbose=True,
)

## 定义任务

In [ ]:
research_task = Task(
    description="总结开源 LLM 开发的三大趋势。",
    agent=research_analyst,
    expected_output="以要点列出三大趋势，并简要解释。",
)

outline_task = Task(
    description="根据研究，为 CTO 生成文章提纲。",
    agent=content_strategist,
    expected_output="包含标题、章节与关键要点的提纲。",
)

review_task = Task(
    description="评审提纲的质量与一致性。",
    agent=content_reviewer,
    expected_output="已评审的提纲，附建议或批准。",
)

## 创建 Crew

In [ ]:
crew = Crew(
    agents=[research_analyst, content_strategist, content_reviewer],
    tasks=[research_task, outline_task, review_task],
    process=Process.sequential,
    verbose=True,
    full_output=True,
)

result = crew.kickoff()
print(result)

### 在 Phoenix 项目中查看运行产生的追踪与跨度。